## Multi-turn AI tutor with function calling for Wikipedia and calculator tools

model used = Qwen/Qwen2.5-3B-Instruct

In [29]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage
import wikipedia
from langchain_groq import ChatGroq  # ← change import



In [30]:
load_dotenv()

True

In [ ]:
# replace the llm + chat_model block with just this:
chat_model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3,
    max_tokens=512,
    api_key="lol i wouldnt do that :D "
)

### Tool 1 : Calculator


In [32]:
def calc(expression):

    try:
        return str(eval(expression))

    except Exception:
        return "Invalid Expression"

### Tool 2 : Wikipedia Search
Searches Wikipedia and returns a short summary.

 Example:
 "Alan Turing"

 Returns:
 Alan Turing was a British mathematician...

In [33]:
def wiki(topic):
    try:
        return wikipedia.summary(
    topic,
    sentences=3
        )       
    except Exception:
        return "No source found"

### Implimenting Chat History
Stores previous messages.

 Multi-turn means:

 User: Who is Alan Turing?
 AI: ...

 User: When was he born?
 AI remembers previous context.

In [34]:
chat_history = []

In [ ]:
print("AI Tutor Started")
print("Type 'exit' to quit.\n")
while True :
    user_input = input("You : ")
    if user_input.lower() == "exit":
        break


 # ------------------------------------------------------
    # Tool Selection Logic
    # ------------------------------------------------------
    #
    # We ask the LLM:
    #
    # Which tool should be used?
    #
    # Allowed tools:
    # calculator
    # wikipedia
    # none
    #
    # ------------------------------------------------------



    tool_prompt = f"""

    You are a tool selector.

    Available tools:

    1. calculator
    Use for mathematical calculations.

    2. wikipedia
    Use for factual topics, people, places,
    inventions, concepts, history.

    3. none
    Use if no tool is needed.

    User Question:
    {user_input}

    Reply with only:

    calculator
    or
    wikipedia
    or
    none

    """
    tool_response = chat_model.invoke(
        [HumanMessage(content=tool_prompt)]
        )
    selected_tool = tool_response.content.strip().lower()


# execute tool

    tool_result = ""
    if "calculator" in selected_tool :
        tool_result = calc(user_input)
    elif "wikipedia" in selected_tool : 
        tool_result = wiki(user_input)
    

    # ------------------------------------------------------
    # Final Prompt
    # ------------------------------------------------------
    #
    # If tool output exists,
    # provide it to the LLM.
    #
    # Otherwise answer normally.
    # ------------------------------------------------------

    final_prompt = f"""

You are a helpful AI Tutor.

Previous Conversation:
{chat_history}

User Question:
{user_input}

Tool Output:
{tool_result}

Answer the user clearly and concisely.

"""

    response = chat_model.invoke(
        [HumanMessage(content=final_prompt)]
    )

    print("\nTutor:", response.content)
    print()

    # ------------------------------------------------------
    # Store Conversation
    # ------------------------------------------------------

    chat_history.append(
        HumanMessage(content=user_input)
    )

    chat_history.append(
        response.content
    )




AI Tutor Started
Type 'exit' to quit.


Tutor: The answer to 5*8 is 40.


Tutor: The capital of India is New Delhi.

